In [ ]:
!pip install -q ultralytics pydicom opencv-python-headless

In [ ]:
import os
import cv2
import yaml
import random
import shutil
import pydicom
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

# 1.配置路径

In [ ]:
BASE_INPUT = Path("/kaggle/input/competitions/rsna-pneumonia-detection-challenge")
TRAIN_DICOM_DIR = BASE_INPUT / "stage_2_train_images"
LABEL_CSV = BASE_INPUT / "stage_2_train_labels.csv"
WORK_DIR = Path("/kaggle/working/rsna_yolo")
IMG_SIZE = 512

TRAIN_IMG_DIR = WORK_DIR / "images" / "train"
VAL_IMG_DIR   = WORK_DIR / "images" / "val"
TRAIN_LBL_DIR = WORK_DIR / "labels" / "train"
VAL_LBL_DIR   = WORK_DIR / "labels" / "val"

for p in [TRAIN_IMG_DIR, VAL_IMG_DIR, TRAIN_LBL_DIR, VAL_LBL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

In [ ]:
df = pd.read_csv(LABEL_CSV)
df.head()

# 2.按照patientId聚合标注

In [ ]:
grouped = df.groupby("patientId")

records = []

for patient_id, g in grouped:
    boxes = []
    has_target = False
    
    for _, row in g.iterrows():
        if int(row["Target"]) == 1:
            has_target = True
            boxes.append([
                float(row["x"]),
                float(row["y"]),
                float(row["width"]),
                float(row["height"])
            ])
    
    records.append({
        "patientId": patient_id,
        "boxes": boxes,
        "target": 1 if has_target else 0
    })

records_df = pd.DataFrame(records)
records_df.head()

# 3.划分训练集/验证集

In [ ]:
train_df, val_df = train_test_split(
    records_df,
    test_size=0.2,
    random_state=42,
    stratify=records_df["target"]
)

print("train:", len(train_df))
print("val:", len(val_df))

# 4.预处理

In [ ]:
def dicom_to_array(dicom_path):
    ds = pydicom.dcmread(str(dicom_path))
    img = ds.pixel_array.astype(np.float32)

    # 归一化到 0~255
    img = img - img.min()
    if img.max() > 0:
        img = img / img.max()
    img = (img * 255).astype(np.uint8)
    
    return img

In [ ]:
def preprocess_image(gray_img, img_size=512):
    """
    输入: 灰度图 uint8
    输出:
      rgb_img_resized: 512x512x3
      scale_x, scale_y: bbox缩放比例
      orig_w, orig_h: 原图尺寸
    """
    orig_h, orig_w = gray_img.shape[:2]

    # CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray_clahe = clahe.apply(gray_img)

    # 灰度转RGB
    rgb_img = cv2.cvtColor(gray_clahe, cv2.COLOR_GRAY2RGB)

    # resize
    rgb_img_resized = cv2.resize(rgb_img, (img_size, img_size))

    scale_x = img_size / orig_w
    scale_y = img_size / orig_h

    return rgb_img_resized, scale_x, scale_y, orig_w, orig_h



In [ ]:
def xywh_to_yolo(x, y, w, h, img_w, img_h):
    x_center = (x + w / 2.0) / img_w
    y_center = (y + h / 2.0) / img_h
    w_norm = w / img_w
    h_norm = h / img_h
    return x_center, y_center, w_norm, h_norm

In [ ]:
def flip_boxes_horizontally(boxes, img_w):
    flipped = []
    for x, y, w, h in boxes:
        new_x = img_w - (x + w)
        flipped.append([new_x, y, w, h])
    return flipped

# 5.生成YOLO数据集

In [ ]:
def save_yolo_sample(patient_id, boxes, image_dir, label_dir, augment_flip=False):
    dicom_path = TRAIN_DICOM_DIR / f"{patient_id}.dcm"
    if not dicom_path.exists():
        return
    
    gray_img = dicom_to_array(dicom_path)
    processed_img, scale_x, scale_y, orig_w, orig_h = preprocess_image(gray_img, IMG_SIZE)

    # 缩放 bbox 到 512x512
    resized_boxes = []
    for x, y, w, h in boxes:
        rx = x * scale_x
        ry = y * scale_y
        rw = w * scale_x
        rh = h * scale_y
        resized_boxes.append([rx, ry, rw, rh])

    # 保存原图
    img_name = f"{patient_id}.png"
    lbl_name = f"{patient_id}.txt"

    cv2.imwrite(str(image_dir / img_name), cv2.cvtColor(processed_img, cv2.COLOR_RGB2BGR))

    with open(label_dir / lbl_name, "w") as f:
        for x, y, w, h in resized_boxes:
            xc, yc, wn, hn = xywh_to_yolo(x, y, w, h, IMG_SIZE, IMG_SIZE)
            # 类别只有1类：pneumonia -> class 0
            f.write(f"0 {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}\n")

    # 训练集可做水平翻转增强
    if augment_flip:
        flipped_img = cv2.flip(processed_img, 1)
        flipped_boxes = flip_boxes_horizontally(resized_boxes, IMG_SIZE)

        flip_img_name = f"{patient_id}_flip.png"
        flip_lbl_name = f"{patient_id}_flip.txt"

        cv2.imwrite(str(image_dir / flip_img_name), cv2.cvtColor(flipped_img, cv2.COLOR_RGB2BGR))

        with open(label_dir / flip_lbl_name, "w") as f:
            for x, y, w, h in flipped_boxes:
                xc, yc, wn, hn = xywh_to_yolo(x, y, w, h, IMG_SIZE, IMG_SIZE)
                f.write(f"0 {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}\n")

In [ ]:
train_pos = train_df[train_df["target"] == 1].reset_index(drop=True)
val_pos   = val_df[val_df["target"] == 1].reset_index(drop=True)

print(len(train_pos), len(val_pos))

In [ ]:
for _, row in train_pos.iterrows():
    save_yolo_sample(
        patient_id=row["patientId"],
        boxes=row["boxes"],
        image_dir=TRAIN_IMG_DIR,
        label_dir=TRAIN_LBL_DIR,
        augment_flip=True
    )

for _, row in val_pos.iterrows():
    save_yolo_sample(
        patient_id=row["patientId"],
        boxes=row["boxes"],
        image_dir=VAL_IMG_DIR,
        label_dir=VAL_LBL_DIR,
        augment_flip=False
    )

# 6.写data.yaml

In [ ]:
data_yaml = {
    "path": str(WORK_DIR),
    "train": "images/train",
    "val": "images/val",
    "nc": 1,
    "names": ["pneumonia"]
}

with open(WORK_DIR / "data.yaml", "w") as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print((WORK_DIR / "data.yaml").read_text())

# 7.训练

In [ ]:
model = YOLO("yolo11s.pt")

model.train(
    data=str(WORK_DIR / "data.yaml"),
    epochs=100,
    imgsz=512,
    batch=16,
    device=0,
    project="/kaggle/working/yolo_rsna",
    name="exp",
    hsv_h=0.0,   
    hsv_s=0.0,
    hsv_v=0.0,
    translate=0.0,
    scale=0.1,
    fliplr=0.0,  
    mosaic=0.0   
)

In [ ]:
metrics = model.val()
print(metrics)

In [ ]:
best_model = YOLO("/kaggle/working/yolo_rsna/exp/weights/best.pt")

sample_img = list(VAL_IMG_DIR.glob("*.png"))[0]
results = best_model.predict(str(sample_img), imgsz=512, conf=0.25, save=True)

print(sample_img)